<a href="https://colab.research.google.com/github/DerWeiseTeufel/MVM_2025/blob/main/benchmarksRusGidro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Benchmark = 25% clean, 25% not full, 25% replaced, 25% synonems.

# Loading the dataset

In [ ]:
from google.colab import userdata


In [ ]:
import pandas as pd
path = userdata.get('path_to_dataset')

# Предположим, у вас есть DataFrame с товарами
df_products = pd.read_csv(path)
df_products.head()

,NameFull
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...


## Test set

In [ ]:
test_df = df_products.sample(500)
test_df.head()

,NameFull
33172,Конденсатор пусковой 35мкФ; 450В СВВ60-Е
30253,Выключатель автоматический IEK ВА 47-29 MVA20-...
11189,Труба электросварная прямошовная 108х4.5 9000м...
15593,Вал карданный задний RS97135.04.02 816мм ГАЗ
41033,Саморез по металлу СММ 4.2х50мм острый


## Perfect set

In [ ]:
perfect = test_df.sample(125)['NameFull']
perfect.head()

,NameFull
45983,"Колесо для тачки FIT 77556 16""x4"""
16660,Комплект колодок тормозные дисковые передние A...
10302,Труба бесшовная горячедеформированная В; с про...
8247,Теплообменник кожухотрубчатый горизонтальный Д...
3068,Устройство дистанционной защиты Schneider Elec...


In [ ]:
rest = test_df.drop(perfect.index)
rest.head()

,NameFull
33172,Конденсатор пусковой 35мкФ; 450В СВВ60-Е
30253,Выключатель автоматический IEK ВА 47-29 MVA20-...
11189,Труба электросварная прямошовная 108х4.5 9000м...
15593,Вал карданный задний RS97135.04.02 816мм ГАЗ
41033,Саморез по металлу СММ 4.2х50мм острый


## synonems

In [ ]:
synonem = rest.sample(125)['NameFull']
synonem

,NameFull
18413,Свеча зажигания LZTR4A-11 NGK
22027,Кольцо каретки подвески уплотнительное 54.31.4...
16206,Втулка выпускного клапана направляющая 4022-10...
27707,Светильник светодиодный настенный/потолочный П...
20740,Корпус кулака поворотного правый 4310-2304030 ...
...,...
19135,Патрубок водяного насоса 50-1307044 МТЗ
11294,"Труба стальная с фаской на концах SMLS XXS 3"" ..."
19227,Блок шестерен промежуточного вала коробки пере...
13796,Шпилька фланцевая сталь тип А; поле доп.6g; дл...


In [ ]:
replaces_synonems = {"двигатель":"мотор",
 "колба":"пробирка",
 "пластик":"пластмасса",
 "труба":"шланг",
 "конфета":"сладость",
 "значок":"знак",
 "топливо":"горючее",
  "сапог":"ботинки",

 }

In [ ]:
# morphological parser - if the parsed(word) == key -> replace(word, map[parsed(word)])

In [ ]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 73.0 MB/s eta 0:00:00


In [ ]:
sentences = synonem
sentences

,NameFull
41674,Фуфайка-свитер термостойкая 88-92/182-188 Т/Б-...
41078,Комбинезон спасателя всесезонный защитный мужс...
23879,Комплект вкладышей двигателя ЯМЗ-238 коренных ...
27521,Колба стекло Кьельдаля 1 1000см3 ТС ГОСТ 25336...
25711,Масло моторное для дизельных двигателей Rimula...
...,...
49841,Зажим шлейфовый соединительный Триас ШС-18.8-1...
17961,Насос топливный высокого давления 70515453 Sha...
40903,Трафарет Цифры пластик
42555,Блок горячеоцинкованных МК стоек под трехфазны...


In [ ]:
replaces_synonems

{'двигатель': 'мотор',
 'колба': 'пробирка',
 'пластик': 'пластмасса',
 'труба': 'шланг',
 'конфеты': 'сладость',
 'значок': 'знак',
 'топливо': 'горючее'}

In [ ]:
import pymorphy3
import pandas as pd
from typing import Dict

def replace_morph_words(series: pd.Series, replacements: Dict[str, str]) -> pd.Series:
    """
    Заменяет слова в предложениях, чья нормальная форма совпадает с ключами словаря.

    Args:
        series: pd.Series со строками-предложениями
        replacements: словарь {нормальная_форма_слова: замена}

    Returns:
        pd.Series с замененными словами
    """
    morph = pymorphy3.MorphAnalyzer()

    def replace_in_text(text: str) -> str:
        if not isinstance(text, str):
            return text

        words = text.lower().split()
        result_words = []

        for word in words:
            # Получаем нормальную форму слова
            parsed = morph.parse(word)[0]
            normal_form = parsed.normal_form

            # Проверяем, есть ли замена для этой нормальной формы
            if normal_form in replacements:
                # Сохраняем регистр и пунктуацию оригинала
                replacement = replacements[normal_form]
                if word[0].isupper():
                    replacement = replacement.capitalize()
                result_words.append(replacement)
            else:
                result_words.append(word)

        return ' '.join(result_words)

    return series.apply(replace_in_text)


# Пример использования
# Тестовые данные
sentences = synonem['NameFull']

# Словарь замен (нормальная_форма -> замена)
replacements = replaces_synonems

# Замена слов
result = replace_morph_words(sentences, replacements)

print("Оригинальные предложения:")
for s in sentences:
    print(f"  {s}")

print("\nПосле замены:")
for s in result:
    print(f"  {s}")

Оригинальные предложения:
  Фуфайка-свитер термостойкая 88-92/182-188 Т/Б-3 Энергоконтракт
  Комбинезон спасателя всесезонный защитный мужской 96-100/170
  Комплект вкладышей двигателя ЯМЗ-238 коренных 238-1000102-В2-Р1 109.75мм КрАЗ;МАЗ
  Колба стекло Кьельдаля 1 1000см3 ТС ГОСТ 25336-82 1-1000-45/40
  Масло моторное для дизельных двигателей Rimula R5 E CF/CF-4/CG-4/CH-4/CI-4 E3/E5/E7 Shell 10W40 канистра 4л
  Наматрасник ткань Hevli City mattress Aqua Protect белый 1200х1950мм
  Ремкомплект натяжного колеса Т-170/130 1409 ЧТЗ
  Грунтовка 20кг KNAUF 545454
  Электрод LB-52U 3.2мм для сварки углеродистых сталей
  Патрон высоковольтный ООО КЭАЗ ПТ 1.1-6-20-40 258471 6000В 20А для защиты электрооборудования У1 ТУ 3414-016-05755766-2007
  Контактор электромагнитный Schneider Electric LC1E3210M5 TeSys E 220В 32А IP20 трехполюсной; 1НО
  Труба бесшовная горячедеформированная В 152х14 Ст20 ГОСТ 8732-78; ГОСТ 8731-74
  Антифриз сине-зеленый Maintain Fricofin Fuchs 600659455 -80t бутыль 1л
  С

In [ ]:
synonem_result = result
synonem_result.head()

,NameFull
41674,фуфайка-свитер термостойкая 88-92/182-188 т/б-...
41078,комбинезон спасателя всесезонный защитный мужс...
23879,комплект вкладышей мотор ямз-238 коренных 238-...
27521,пробирка стекло кьельдаля 1 1000см3 тс гост 25...
25711,масло моторное для дизельных мотор rimula r5 e...


In [ ]:
sentences.apply(replace_in_text)

,NameFull
41674,Фуфайка-свитер термостойкая 88-92/182-188 Т/Б-...
41078,Комбинезон спасателя всесезонный защитный мужс...
23879,Комплект вкладышей двигателя ЯМЗ-238 коренных ...
27521,Колба стекло Кьельдаля 1 1000см3 ТС ГОСТ 25336...
25711,Масло моторное для дизельных двигателей Rimula...
...,...
49841,Зажим шлейфовый соединительный Триас ШС-18.8-1...
17961,Насос топливный высокого давления 70515453 Sha...
40903,Трафарет Цифры пластик
42555,Блок горячеоцинкованных МК стоек под трехфазны...


## Random change of chars

In [ ]:
rest = rest.drop(synonem.index)


In [ ]:
verschreibung = rest.sample(125)

In [ ]:
import pandas as pd
import numpy as np
import random

def replace_chars_random(series, percent=1, n_chars=1, seed=None):
    """Простая версия для случайной замены символов"""
    if seed:
        random.seed(seed)
        np.random.seed(seed)

    result = series.copy()
    n_modify = max(1, int(len(series) * percent))
    idx_modify = np.random.choice(series.index, n_modify, replace=False)

    for idx in idx_modify:
        text = str(series[idx]) if pd.notna(series[idx]) else ""
        if len(text) < 2:
            continue

        # Заменяем n_chars случайных символов
        for _ in range(min(n_chars, len(text))):
            pos = random.randint(0, len(text)-1)
            # Заменяем на случайную букву/цифру
            text = text[:pos] + random.choice('абвгдежзиклмнопрстуфхцчшщъыьэюя0123456789') + text[pos+1:]

        result[idx] = text

    return result

# Быстрый пример
data = verschreibung['NameFull']
print("До:", list(data))
data_after_1 = replace_chars_random(data, percent=1, n_chars=1, seed=42)
print("После:", list(data_after_1))
data_after_2 = replace_chars_random(data_after_1, percent=0.75, n_chars=1, seed=42)
print("После:", list(data_after_2))
data_after_3 = replace_chars_random(data_after_2, percent=0.5, n_chars=2, seed=42)
print("После:", list(data_after_3))


До: ['Насос гидроусилителя руля 5286672F ГАЗ', 'Стойка стабилизатора передняя правая 3102-2904056 ГАЗ ГАЗ', 'Говядина бескостная котлетное мясо', 'Плита нагревательная односекционная Экросхим ES-H3040', 'Отвод для внутренней канализации PP 67гр 110мм раструб серый', 'Автошина внедорожная TL зимняя нешипованная Bridgestone Blizzak DM-V2 285мм/65/R17 116R', 'Кабель силовой с изоляцией из ПВХ ВВГ-Пнг(A)-LS 2x2.5-0.66кВ', 'Венок из искуственных цветов', 'Источник бесперебойного питания общего назначения с функцией подключения внешней батареи Бастион SKAT-24-2.0-DIN 220/24В AC/DCВ 187…242В 70ВА 2А DIN-рейка IP20 139х89х65мм', 'Труба бесшовная горячедеформированная В 57х3.5 КР 4000мм Ст20 ГОСТ 8732-78; ГОСТ 8731-74', 'Футболка спортивная летняя мужская полиэстер синий XL короткий ТР ТС 017/2011 Asics SS Tee Indoor 2 149126 0861', 'Текущий ремонт генератора синхронного к гидравлической турбине вертикальной Г-2 Новотроицкой ГЭС', 'Ограждение барьерное 1.5м 60мм 1040мм пластик красное Argo с ко

## Random removal of words

In [ ]:
from typing import Optional
# Упрощенная версия с логикой "нельзя удалять больше чем n-3 слов"
def drop_words_simple(
    series: pd.Series,
    keep_at_least: int = 3,
    drop_chance: float = 0.8,
    random_state: Optional[int] = None
) -> pd.Series:
    """
    Простая версия: в drop_chance случаев удаляет случайное количество слов,
    но оставляет не менее keep_at_least слов.
    """
    if random_state is not None:
        np.random.seed(random_state)

    result = series.copy()

    for idx, text in series.items():
        if pd.isna(text) or not isinstance(text, str):
            continue

        words = text.split()
        n_words = len(words)

        # Пропускаем если слов меньше минимального
        if n_words <= keep_at_least:
            continue

        # Решаем, удалять ли слова
        if np.random.random() > drop_chance:
            continue

        # Максимально можно удалить n_words - keep_at_least слов
        max_drop = n_words - keep_at_least

        # Выбираем сколько удалить (1..max_drop)
        n_to_drop = np.random.randint(1, max_drop + 1)

        # Выбираем слова для удаления
        indices_to_remove = np.random.choice(n_words, size=n_to_drop, replace=False)
        new_words = [word for i, word in enumerate(words) if i not in indices_to_remove]

        result[idx] = ' '.join(new_words)

    return result

In [ ]:
to_drop = rest.sample(125)
after_drop = drop_words_simple(to_drop['NameFull'])
after_drop

,NameFull
11990,Тройник электросварной ПЭ100 под сварку
16709,Выключатель габаритных огней клавишный 581.371...
7692,Корпус насоса Н14.2.933.01.006 01/007.01 для Г...
21609,Катушка Б114-Б ГАЗ;ЗИЛ;ПАЗ
10986,Труба бесшовная горячедеформированная В 60х5 0...
...,...
38336,переплетная Yunger M168 PRO 1000лист А3
43074,металлу; работ белая
44727,Изолятор подвесной стеклянный ПС-120Б 112W
1931,Термопреобразователь сопротивления ГК Промприб...


## Merging the results

In [ ]:
type(perfect)

pandas.core.series.Series

In [ ]:
type(data_after_3)

pandas.core.series.Series

In [ ]:
type(after_drop)

pandas.core.series.Series

In [ ]:
test_df_preproccess = pd.concat([perfect, data_after_3,after_drop, synonem])
test_df_preproccess

,NameFull
45983,"Колесо для тачки FIT 77556 16""x4"""
16660,Комплект колодок тормозные дисковые передние A...
10302,Труба бесшовная горячедеформированная В; с про...
8247,Теплообменник кожухотрубчатый горизонтальный Д...
3068,Устройство дистанционной защиты Schneider Elec...
...,...
19135,Патрубок водяного насоса 50-1307044 МТЗ
11294,"Труба стальная с фаской на концах SMLS XXS 3"" ..."
19227,Блок шестерен промежуточного вала коробки пере...
13796,Шпилька фланцевая сталь тип А; поле доп.6g; дл...


In [ ]:
test_df_preproccess = test_df_preproccess.reset_index()
test_df_preproccess

,index,NameFull
0,45983,"Колесо для тачки FIT 77556 16""x4"""
1,16660,Комплект колодок тормозные дисковые передние A...
2,10302,Труба бесшовная горячедеформированная В; с про...
3,8247,Теплообменник кожухотрубчатый горизонтальный Д...
4,3068,Устройство дистанционной защиты Schneider Elec...
...,...,...
495,19135,Патрубок водяного насоса 50-1307044 МТЗ
496,11294,"Труба стальная с фаской на концах SMLS XXS 3"" ..."
497,19227,Блок шестерен промежуточного вала коробки пере...
498,13796,Шпилька фланцевая сталь тип А; поле доп.6g; дл...


In [ ]:
test_df_preproccess.to_csv('test_df.csv', index=False)